# Chapter 10C — Stage C: Multimodal Waveform Debugger

**Multi-Agent Analog EDA — Pedagogical Project (PhD track)**

---

**Stage C** closes a *perceptual* loop: an agent **renders** analog verification waveforms (Bode and transient) as **images**, then performs **multimodal reasoning**—the same workflow enabled by **vision-capable LLMs** (e.g., **Gemini 3 Pro**, **Claude 4.5 Vision**)—to **hypothesize defects** and **propose bias/compensation edits**.

### Learning objectives

1. **Formalize** analog debug as *joint inference* over $(\text{waveform image}, \text{spec}, \text{netlist metadata}) \rightarrow \text{repair action}$.
2. **Synthesize** **self-consistent** healthy/faulty **Bode** and **step** datasets using **low-order macromodels** (no external simulator).
3. **Implement** a **numeric waveform analysis engine**: **gain/phase margins**, **Nyquist-style summaries**, **settling/overshoot**, and **THD** via **FFT**.
4. **Build** **annotated diagnostic figures** (matplotlib, dark theme) with **pass/fail** bands and **spec vs. actual** juxtaposition.
5. **Prototype** a **vision-agent interface**: **structured prompts**, **PNG ingestion** (simulated), **deterministic** “LLM” replies, and **parsing** of bias/compensation suggestions.
6. **Run** a **multi-iteration debug loop** with **convergence tracking** (plotly, dark template).

### Notation

- Open-loop gain $L(s)$; unity-gain frequency $\omega_{\mathrm{gc}}$; phase margin $\mathrm{PM}$.
- Closed-loop transfer $T(s)$ for transient studies; settling time $t_s$, overshoot $M_p$.
- Total harmonic distortion $\mathrm{THD}$ computed from discrete spectra.

> **Disclaimer:** No cloud API keys are required. **LLM vision calls are simulated** with rule-based responses tied to extracted metrics—swap `simulate_vision_llm` for a real multimodal client in production.

---

In [ ]:
# Imports, reproducibility, dark-theme defaults (matplotlib #0d1117, plotly plotly_dark)
from __future__ import annotations

import io
import re
import json
import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy import signal
from scipy.interpolate import interp1d

import plotly.graph_objects as go
import plotly.io as pio

RNG = np.random.default_rng(2026)

DARK_BG = "#0d1117"
ACCENT = "#58a6ff"
GREEN = "#3fb950"
AMBER = "#d29922"
RED = "#f85149"
PURPLE = "#bc8cff"
CYAN = "#79c0ff"
MAGENTA = "#ff7b72"

MPL_RC = {
    "figure.facecolor": DARK_BG,
    "axes.facecolor": DARK_BG,
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": "#c9d1d9",
    "text.color": "#c9d1d9",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "grid.color": "#21262d",
    "grid.alpha": 0.65,
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "font.size": 11,
}
mpl.rcParams.update(MPL_RC)
pio.templates.default = "plotly_dark"

print("numpy", np.__version__, "| scipy", end=" ")
import scipy as _sp
print(_sp.__version__, "| matplotlib", mpl.__version__)
print("plotly default template:", pio.templates.default)

## 10C.1 Problem statement — why *vision* matters in analog debug

**Analog verification is visual.** Designers routinely inspect **Bode plots** (stability, bandwidth, peaking) and **transients** (ringing, slew, settling) to localize **bias**, **compensation**, and **loading** issues. Text-only logs (`.print` lines) discard **shape cues** that humans—and **multimodal models**—exploit.

**Multimodal LLMs** (e.g., **Gemini 3 Pro**, **Claude 4.5 Vision**) can ingest **rendered plots** and align them with **spec sheets**, highlighting **margin violations**, **unexpected resonances**, and **harmonic content**. In agentic EDA, the pattern is:

$$\text{simulate} \;\rightarrow\; \text{rasterize} \;\rightarrow\; \text{vision analysis} \;\rightarrow\; \text{structured repair} \;\rightarrow\; \text{re-simulate}.$$

**Stage C objective:** implement this loop on **synthetic** waveforms, with a **numeric oracle** (Python metrics) grounding **simulated** vision responses.

**Deliverables in this notebook**

| Artifact | Role |
|----------|------|
| Synthetic Bode / step data | Train intuition for fault signatures |
| Numeric metrics engine | Objective checks (margins, THD, settling) |
| Annotated PNG figures | Inputs to (simulated) multimodal reasoning |
| Prompt + parser | Contracts between tools and LLMs |
| Iteration trace | Study convergence of closed-loop debug |

In [ ]:
# --- Synthetic Bode data: open-loop L(s) macromodels ---------------------------------

def _poly_from_poles_dc(A0: float, poles_rps: List[float]) -> Tuple[np.ndarray, np.ndarray]:
    """Return (num, den) for L(s)=A0 * prod(wp_k)/(prod(s+wp_k))."""
    den = np.array([1.0])
    gain = float(A0)
    for wp in poles_rps:
        gain *= wp
        den = np.convolve(den, [1.0, float(wp)])
    num = np.array([gain])
    return num, den


def bode_open_loop(
    A0: float,
    poles_rps: List[float],
    w: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    num, den = _poly_from_poles_dc(A0, poles_rps)
    sys = signal.TransferFunction(num, den)
    w_rad_s, mag, phase = signal.bode(sys, w=w)
    return w_rad_s, mag, phase  # mag in dB, phase in deg


def add_rhp_zero_effect(phase_deg: np.ndarray, w_rad: np.ndarray, wz: float, kappa: float = 1.0) -> np.ndarray:
    """Approximate extra phase lag from a far RHP zero: -2*atan(w/wz) in deg (scaled by kappa)."""
    return phase_deg - kappa * (2.0 * np.degrees(np.arctan(np.maximum(w_rad, 1e-12) / max(wz, 1e-12))))


def synthesize_bode_scenarios() -> Dict[str, Dict[str, np.ndarray]]:
    """Healthy vs. common loop faults (low PM, gain peaking via effective underdamped closed-loop proxy)."""
    w = np.logspace(1, 9, 2000)  # rad/s

    # Healthy: dominant pole + benign second pole far beyond crossover
    w_gc_healthy = 2 * np.pi * 12e6
    pm_target = 65.0
    # pick poles so PM ~ 65° at unity (engineering sketch)
    p1 = w_gc_healthy / (10 ** (80 / 20))  # ~120 krad/s if A0=80dB
    A0_db = 80.0
    A0 = 10 ** (A0_db / 20)
    p2_healthy = 8 * w_gc_healthy

    w_h, mag_h, ph_h = bode_open_loop(A0, [p1, p2_healthy], w)
    healthy = {"w": w_h, "mag_db": mag_h, "phase_deg": ph_h}

    # Low phase margin: bring second pole closer + mild RHP-zero phase erosion
    p2_bad = 1.2 * w_gc_healthy
    w_l, mag_l, ph_l = bode_open_loop(A0, [p1, p2_bad], w)
    ph_l = add_rhp_zero_effect(ph_l, w_l, wz=3 * w_gc_healthy, kappa=0.85)
    low_pm = {"w": w_l, "mag_db": mag_l, "phase_deg": ph_l}

    # Gain peaking: model as closed-loop resonance viewed on T(s) magnitude (not strictly L(s))
    wn = 2 * np.pi * 15e6
    zeta = 0.35
    num_t = [wn**2]
    den_t = [1, 2 * zeta * wn, wn**2]
    sys_t = signal.TransferFunction(num_t, den_t)
    wt, magt, pht = signal.bode(sys_t, w=w)
    gain_peak = {"w": wt, "mag_db": magt, "phase_deg": pht, "note": "closed_loop_T(s) peaking"}

    # "Oscillation risk": very low PM + slight bump near unity (use aggressive second pole + RHP)
    p2_os = 0.55 * w_gc_healthy
    w_o, mag_o, ph_o = bode_open_loop(A0, [p1, p2_os], w)
    ph_o = add_rhp_zero_effect(ph_o, w_o, wz=1.1 * w_gc_healthy, kappa=1.0)
    osc_risk = {"w": w_o, "mag_db": mag_o, "phase_deg": ph_o}

    return {
        "healthy_open_loop": healthy,
        "low_phase_margin": low_pm,
        "gain_peaking_closed_loop": gain_peak,
        "oscillation_risk_open_loop": osc_risk,
    }


BODE_SET = synthesize_bode_scenarios()
print("Scenarios:", list(BODE_SET.keys()))

## 10C.2 Synthetic transients — step response and settling faults

We study a **second-order** closed-loop prototype

$$T(s)=\frac{\omega_n^2}{s^2+2\zeta\omega_n s+\omega_n^2},$$

which captures **overshoot** ($\zeta$ small), **ringing** ($\zeta$ very small), and **approximate oscillation** (very lightly damped). We also inject a **harmonic residual** to exercise **THD** estimation.

**Fault taxonomy (pedagogical)**

| Fault | Signature | Mechanistic sketch |
|-------|-----------|--------------------|
| **Excessive overshoot** | Large $M_p$, short rise | Low $\zeta$ |
| **Ringing / instability** | Persistent oscillation envelope | Very low $\zeta$ |
| **Oscillation** | Narrowband spectral line | Added sinusoid |
| **Settling fault** | Long $t_s$ | Lower $\zeta$ + smaller $\omega_n$ |

The next cell **simulates** `scipy.signal.step` responses and stores **metadata** for labeling.

In [ ]:
# --- Synthetic step / transient datasets ------------------------------------------------

def second_order_step(wn: float, zeta: float, t: np.ndarray) -> np.ndarray:
    num = [wn**2]
    den = [1.0, 2 * zeta * wn, wn**2]
    sys = signal.TransferFunction(num, den)
    t_out, y = signal.step(sys, T=t)
    return t_out, y


def add_harmonic_distortion(y: float, t: float, f0: float, amps: Dict[int, float]) -> float:
    y2 = y
    for k, ak in amps.items():
        y2 += ak * np.sin(2 * np.pi * k * f0 * t)
    return y2


def synthesize_transient_scenarios(
    t_max: float = 5e-7,
    n: int = 8000,
) -> Dict[str, Dict[str, np.ndarray]]:
    t = np.linspace(0.0, t_max, n, endpoint=False)
    f0 = 50e6  # 50 MHz fundamental for THD toy tone

    wn = 2 * np.pi * 35e6
    zeta_ok = 0.78
    t_o, y_o = second_order_step(wn, zeta_ok, t)

    zeta_os = 0.28
    t_os, y_os = second_order_step(wn, zeta_os, t)

    wn_slow = 2 * np.pi * 18e6
    zeta_ring = 0.09
    t_r, y_r = second_order_step(wn_slow, zeta_ring, t)

    # oscillation: strong harmonic residual at f0 and 3f0
    y_osc = np.array([add_harmonic_distortion(float(y), float(tt), f0, {1: 0.02, 3: 0.012}) for y, tt in zip(y_r, t_r)])

    # healthy with mild distortion (THD ~ few %)
    y_thd = np.array([add_harmonic_distortion(float(y), float(tt), f0, {2: 0.018, 3: 0.006}) for y, tt in zip(y_o, t_o)])

    return {
        "healthy_step": {"t": t_o, "y": y_o, "fs": 1.0 / (t[1] - t[0]), "label": "healthy"},
        "excessive_overshoot": {"t": t_os, "y": y_os, "fs": 1.0 / (t[1] - t[0]), "label": "overshoot"},
        "ringing_low_damping": {"t": t_r, "y": y_r, "fs": 1.0 / (t[1] - t[0]), "label": "ringing"},
        "oscillation_spectral": {"t": t_r, "y": y_osc, "fs": 1.0 / (t[1] - t[0]), "label": "oscillation"},
        "mild_nonlinearity_thd": {"t": t_o, "y": y_thd, "fs": 1.0 / (t[1] - t[0]), "label": "thd"},
    }


TRANSIENT_SET = synthesize_transient_scenarios()

fig, ax = plt.subplots(1, 1, figsize=(10, 4), dpi=120)
fig.patch.set_facecolor(DARK_BG)
for name, pack in list(TRANSIENT_SET.items())[:4]:
    ax.plot(pack["t"] * 1e9, pack["y"], lw=1.2, label=name)
ax.set_title("Synthetic step envelopes (first four scenarios)")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Normalized amplitude")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.grid(True)
plt.tight_layout()
plt.show()

## 10C.3 Waveform analysis engine — margins, Nyquist summary, settling, THD

### Gain and phase margins
Given sampled Bode data for **open-loop** $L(j\omega)$:

- **Gain crossover** $\omega_{\mathrm{gc}}$: $|L(j\omega_{\mathrm{gc}})|=1 \Rightarrow 0\,\mathrm{dB}$.
- **Phase margin:** $\mathrm{PM}=180^\circ+\angle L(j\omega_{\mathrm{gc}})$.
- **Phase crossover** $\omega_{\pi}$: $\angle L(j\omega_{\pi})=-180^\circ$ (unwrap-aware).
- **Gain margin (dB):** $\mathrm{GM}=-|L(j\omega_{\pi})|_{\mathrm{dB}}$.

### Nyquist criterion (conceptual bridge)
For **LTI** feedback with rational $L(s)$, closed-loop stability relates to **winding** of the Nyquist contour about $-1$. In practice, **PM/GM** are **conservative surrogates** when the loop is **minimum-phase-ish**; we emit a **qualitative** verdict from margins.

### Settling and overshoot
For a bounded step $y(t)$ targeting $y_\infty$, pick **settling band** $\pm\varepsilon$ (e.g., 2%). **Overshoot** $M_p=\max_t y(t)/y_\infty-1$ for overdamped/underdamped positives.

### THD
Let $Y[k]$ be RMS amplitudes of fundamental $f_0$ and harmonics. Then

$$\mathrm{THD}=\frac{\sqrt{\sum_{k\ge 2} |Y[k]|^2}}{|Y[1]|}.$$

All of the above are implemented as **pure NumPy/SciPy** functions next.

In [ ]:
# --- Numeric metrics --------------------------------------------------------------------

def _interp_x_for_y(x: np.ndarray, y: np.ndarray, y_target: float) -> float:
    """Return x where y crosses y_target (linear interp)."""
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    idx = np.where(np.diff(np.sign(y - y_target)))[0]
    if len(idx) == 0:
        return float("nan")
    i = int(idx[0])
    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]
    t = (y_target - y0) / (y1 - y0 + 1e-18)
    return float(x0 + t * (x1 - x0))


def extract_gain_phase_margins(
    w_rad: np.ndarray,
    mag_db: np.ndarray,
    phase_deg: np.ndarray,
) -> Dict[str, float]:
    w_rad = np.asarray(w_rad, dtype=float)
    mag_db = np.asarray(mag_db, dtype=float)
    phase_deg = np.unwrap(np.deg2rad(np.asarray(phase_deg, dtype=float)))
    phase_deg = np.rad2deg(phase_deg)

    w_gc = _interp_x_for_y(w_rad, mag_db, 0.0)
    if math.isnan(w_gc):
        pm = float("nan")
    else:
        f_mag = interp1d(np.log10(w_rad), mag_db, kind="linear", fill_value="extrapolate")
        f_ph = interp1d(np.log10(w_rad), phase_deg, kind="linear", fill_value="extrapolate")
        pm = 180.0 + float(f_ph(np.log10(w_gc)))

    w_pi = _interp_x_for_y(w_rad, phase_deg, -180.0)
    if math.isnan(w_pi):
        gm_db = float("nan")
    else:
        gm_db = -float(interp1d(np.log10(w_rad), mag_db, kind="linear", fill_value="extrapolate")(np.log10(w_pi)))

    return {"w_gc_rad_s": w_gc, "phase_margin_deg": pm, "w_pi_rad_s": w_pi, "gain_margin_dB": gm_db}


def nyquist_stability_summary(margins: Dict[str, float]) -> Dict[str, object]:
    pm = margins.get("phase_margin_deg", float("nan"))
    gm = margins.get("gain_margin_dB", float("nan"))
    verdict = "indeterminate"
    if (not math.isnan(pm)) and pm > 45 and (not math.isnan(gm)) and gm > 6:
        verdict = "likely_stable_minimum_phase_surrogate"
    elif (not math.isnan(pm)) and pm < 35:
        verdict = "high_risk_low_phase_margin"
    elif (not math.isnan(gm)) and gm < 0:
        verdict = "violation_gain_margin_nonpositive"
    else:
        verdict = "marginal_review_nyquist_detail"
    return {"verdict": verdict, "pm_deg": pm, "gm_dB": gm}


def settling_time_and_overshoot(
    t: np.ndarray,
    y: np.ndarray,
    band: float = 0.02,
) -> Dict[str, float]:
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    y_ss = float(np.mean(y[int(0.92 * len(y)) :]))
    if abs(y_ss) < 1e-9:
        y_ss = 1.0
    yn = y / y_ss
    Mp = float(np.max(yn) - 1.0)
    lo, hi = 1.0 - band, 1.0 + band
    inside = (yn >= lo) & (yn <= hi)
    if not np.any(inside):
        ts = float("nan")
    else:
        last_out = np.max(t[~inside]) if np.any(~inside) else 0.0
        idx_first = np.argmax(inside & (t >= last_out))
        ts = float(t[idx_first])
    return {"y_ss": y_ss, "overshoot": Mp, "settling_time_s": ts, "band": band}


def compute_thd_percent(y: np.ndarray, fs: float, f0: float, max_harm: int = 8) -> Dict[str, float]:
    y = np.asarray(y, dtype=float)
    y = y - np.mean(y)
    win = np.hanning(len(y))
    Y = np.fft.rfft(y * win)
    freqs = np.fft.rfftfreq(len(y), d=1.0 / fs)
    mag = np.abs(Y)
    fund_idx = int(np.argmin(np.abs(freqs - f0)))
    fund = max(mag[fund_idx], 1e-18)
    harm_power = 0.0
    counted = 0
    for k in range(2, max_harm + 1):
        fk = k * f0
        if fk >= freqs[-1]:
            break
        hk = int(np.argmin(np.abs(freqs - fk)))
        harm_power += float(mag[hk] ** 2)
        counted += 1
    thd = 100.0 * math.sqrt(harm_power) / fund
    return {"THD_percent": thd, "fundamental_Hz": f0, "harmonics_used": float(counted)}


# quick self-check on healthy open-loop Bode
m = extract_gain_phase_margins(BODE_SET["healthy_open_loop"]["w"], BODE_SET["healthy_open_loop"]["mag_db"], BODE_SET["healthy_open_loop"]["phase_deg"])
print("Healthy margins:", json.dumps(m, indent=2))
print("Nyquist summary:", nyquist_stability_summary(m))

stp = TRANSIENT_SET["excessive_overshoot"]
so = settling_time_and_overshoot(stp["t"], stp["y"])
print("Overshoot scenario:", json.dumps(so, indent=2))

thd_pack = compute_thd_percent(TRANSIENT_SET["mild_nonlinearity_thd"]["y"], TRANSIENT_SET["mild_nonlinearity_thd"]["fs"], f0=50e6)
print("THD mild:", json.dumps(thd_pack, indent=2))

## 10C.4 Visual diagnostics — annotated plots, pass/fail, spec vs. actual

We render **publication-style** dark figures that mirror what a **human** (or **vision model**) would inspect:

- **Critical frequencies:** $\omega_{\mathrm{gc}}$, $\omega_{\pi}$, **unity-gain** line.
- **Shaded pass bands:** e.g., $\mathrm{PM}\ge 55^\circ$, $\mathrm{GM}\ge 6\,\mathrm{dB}$ (tunable).
- **Side-by-side** panels comparing **target template** vs. **measured** curves.

These annotations **anchor** multimodal prompts: the model can be asked to **read** numeric overlays and **align** them with **spec tables**.

In [ ]:
# --- Annotated matplotlib diagnostics ---------------------------------------------------

SPEC = {
    "PM_min_deg": 55.0,
    "GM_min_dB": 6.0,
    "GBW_target_Hz": 12e6,
    "overshoot_max": 0.15,
    "settling_ns_max": 120.0,
    "THD_max_percent": 2.5,
}


def plot_diagnostic_bode(
    pack: Dict[str, np.ndarray],
    title: str,
    spec: Dict[str, float] = SPEC,
    save_path: Optional[str] = None,
) -> plt.Figure:
    w = pack["w"]
    mag_db = pack["mag_db"]
    phase_deg = pack["phase_deg"]
    margins = extract_gain_phase_margins(w, mag_db, phase_deg)
    w_gc = margins["w_gc_rad_s"]
    w_pi = margins["w_pi_rad_s"]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, dpi=140)
    fig.patch.set_facecolor(DARK_BG)

    # Magnitude
    ax1.semilogx(w / (2 * np.pi), mag_db, color=ACCENT, lw=2.0, label="|L(jω)|")
    ax1.axhline(0, color="#8b949e", lw=1.0, ls="--")
    if not math.isnan(w_gc):
        ax1.axvline(w_gc / (2 * np.pi), color=GREEN, ls="--", lw=1.2, label="f_gc (0 dB)")
    pm_ok = (not math.isnan(margins["phase_margin_deg"])) and margins["phase_margin_deg"] >= spec["PM_min_deg"]
    band_color = GREEN if pm_ok else RED
    ax1.fill_between(w / (2 * np.pi), -120, 0, where=(mag_db <= 0), color=band_color, alpha=0.06)
    ax1.set_ylabel("dB")
    ax1.set_title(title)
    ax1.grid(True, which="both")
    ax1.legend(frameon=False, loc="lower left", fontsize=9)

    # Phase with PM annotation
    ax2.semilogx(w / (2 * np.pi), phase_deg, color=PURPLE, lw=2.0, label="∠L(jω)")
    ax2.axhline(-180, color="#8b949e", lw=1.0, ls="--")
    if not math.isnan(w_gc):
        f_interp = interp1d(np.log10(w), phase_deg, kind="linear", fill_value="extrapolate")
        ph_gc = float(f_interp(np.log10(w_gc)))
        ax2.scatter([w_gc / (2 * np.pi)], [ph_gc], color=AMBER, s=36, zorder=5)
        ax2.annotate(
            f"PM≈{180+ph_gc:.1f}°",
            xy=(w_gc / (2 * np.pi), ph_gc),
            xytext=(w_gc / (2 * np.pi) * 1.8, ph_gc + 22),
            color=CYAN,
            arrowprops=dict(arrowstyle="->", color=CYAN, lw=0.8),
            fontsize=9,
        )
    if not math.isnan(w_pi):
        ax2.axvline(w_pi / (2 * np.pi), color=MAGENTA, ls=":", lw=1.2, label="f_π (−180°)")
    pm_region = np.full_like(phase_deg, -180 + spec["PM_min_deg"])
    ax2.fill_between(w / (2 * np.pi), -180, pm_region, color=GREEN, alpha=0.07, label="PM pass band")
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Degrees")
    ax2.grid(True, which="both")
    ax2.legend(frameon=False, loc="lower left", fontsize=8)

    fig.suptitle(
        f"Diagnostics | GM≈{margins['gain_margin_dB']:.1f} dB | PM≈{margins['phase_margin_deg']:.1f}°",
        color="#c9d1d9",
        y=0.995,
        fontsize=11,
    )
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, facecolor=DARK_BG)
    return fig


def plot_spec_vs_actual_bode(
    actual_pack: Dict[str, np.ndarray],
    template_pack: Dict[str, np.ndarray],
    title: str = "Spec template vs. actual (magnitude)",
) -> plt.Figure:
    fig, ax = plt.subplots(1, 1, figsize=(10, 4), dpi=140)
    fig.patch.set_facecolor(DARK_BG)
    ax.semilogx(template_pack["w"] / (2 * np.pi), template_pack["mag_db"], color=GREEN, lw=2.0, ls="--", label="template (healthy)")
    ax.semilogx(actual_pack["w"] / (2 * np.pi), actual_pack["mag_db"], color=ACCENT, lw=2.0, label="actual (candidate)")
    ax.axhline(0, color="#8b949e", lw=1.0, ls=":")
    ax.axvline(SPEC["GBW_target_Hz"], color=AMBER, ls="--", lw=1.0, label="GBW target")
    ax.set_title(title)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("dB")
    ax.grid(True, which="both")
    ax.legend(frameon=False)
    plt.tight_layout()
    return fig


fig1 = plot_diagnostic_bode(BODE_SET["low_phase_margin"], "Fault: low phase margin (open-loop sketch)")
plt.show()
fig2 = plot_spec_vs_actual_bode(BODE_SET["low_phase_margin"], BODE_SET["healthy_open_loop"])
plt.show()

## 10C.5 Multimodal agent interface — PNG, prompts, simulated vision, parsing

**Vision-capable LLMs** consume **images** (PNG/JPEG) plus **structured text**. A robust agent pattern:

1. **Rasterize** matplotlib/plotly figures to **bytes** (`io.BytesIO`).
2. Attach **spec JSON** and **numeric pre-metrics** (optional *chain-of-tools* grounding).
3. Use a **strict prompt template** requesting **machine-parseable** suggestions.
4. **Parse** model output into `bias_current_scale`, `compensation_cap_pf`, and **free-text rationale**.

Here we **simulate** the multimodal response using **deterministic rules** on extracted metrics—no API keys. Replace `simulate_vision_llm` with a call to your provider’s **vision** endpoint while preserving the **I/O contract**.

In [ ]:
# --- Prompt templates, PNG buffer, simulated multimodal LLM, parsing ----------------------
import textwrap

VISION_ANALYSIS_PROMPT = """You are an analog verification agent with vision access to Bode and transient plots.

Context (JSON):
{{METRICS_JSON}}

Task:
1. Identify the dominant analog fault signature visible in the plot (if any).
2. Propose TWO concrete knob moves: tail bias current I_TAIL (scale factor) and Miller/Cc compensation (picofarads).
3. Return EXACTLY the following machine-readable trailer:

BEGIN_SUGGESTIONS
I_BIAS_SCALE: <float, e.g. 1.08>
CC_PF: <float, e.g. 2.4>
CONFIDENCE: <0..1>
NOTES: <one line>
END_SUGGESTIONS

Rules:
- If phase margin is low, prefer INCREASE CC_PF and/or DECREASE I_BIAS_SCALE modestly.
- If THD is high, prefer INCREASE I_BIAS_SCALE slightly (more gm) and check large-signal slewing (not modeled here).
- If excessive overshoot, prefer INCREASE CC_PF.
"""


def figure_to_png_bytes(fig: plt.Figure) -> bytes:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", facecolor=fig.get_facecolor(), dpi=160)
    plt.close(fig)
    buf.seek(0)
    return buf.read()


def simulate_vision_llm(metrics: Dict[str, object], scenario: str) -> str:
    """Deterministic stand-in for Gemini/Claude vision: uses metrics, ignores real pixels."""
    pm = float(metrics.get("phase_margin_deg", float("nan")))
    thd = float(metrics.get("THD_percent", 0.0))
    ov = float(metrics.get("overshoot", 0.0))

    ib = 1.0
    cc = 1.6
    notes = "No dominant fault; maintain nominal biasing."

    if scenario.lower().startswith("low") or (not math.isnan(pm) and pm < 50):
        cc = min(4.5, cc * 1.35)
        ib = max(0.85, ib * 0.93)
        notes = "Low PM: increase Miller compensation; slightly reduce bias to shift poles."
    if "peaking" in scenario.lower() or (not math.isnan(pm) and pm < 40):
        cc = min(5.5, cc * 1.15)
        notes = "Gain peaking / low damping: push dominant pole separation via Cc."
    if thd > SPEC["THD_max_percent"]:
        ib = min(1.25, ib * 1.08)
        notes = "Elevated THD: increase bias current to improve small-signal linearity (toy model)."
    if ov > SPEC["overshoot_max"]:
        cc = min(6.0, cc * 1.2)
        notes = "Excessive overshoot: raise compensation to dampen secondary pole pair."

    conf = 0.55
    if (not math.isnan(pm)) and pm >= SPEC["PM_min_deg"] and thd <= SPEC["THD_max_percent"] and ov <= SPEC["overshoot_max"]:
        conf = 0.9

    return textwrap.dedent(
        f"""
        (Simulated multimodal analysis)

        The rendered plot and JSON metrics indicate scenario '{scenario}'.
        Dominant hypothesis aligns with automated metrics (PM/THD/overshoot).

        BEGIN_SUGGESTIONS
        I_BIAS_SCALE: {ib:.3f}
        CC_PF: {cc:.3f}
        CONFIDENCE: {conf:.2f}
        NOTES: {notes}
        END_SUGGESTIONS
        """
    ).strip()


def parse_agent_suggestions(text: str) -> Dict[str, object]:
    m = re.search(
        r"BEGIN_SUGGESTIONS\s*I_BIAS_SCALE:\s*([0-9eE+\-.]+)\s*CC_PF:\s*([0-9eE+\-.]+)\s*CONFIDENCE:\s*([0-9eE+\-.]+)\s*NOTES:\s*(.*?)END_SUGGESTIONS",
        text,
        flags=re.S,
    )
    if not m:
        return {"ok": False, "raw": text}
    return {
        "ok": True,
        "I_BIAS_SCALE": float(m.group(1)),
        "CC_PF": float(m.group(2)),
        "CONFIDENCE": float(m.group(3)),
        "NOTES": m.group(4).strip(),
    }


# Demo: build PNG for multimodal payload (bytes), then simulate response
figd = plot_diagnostic_bode(BODE_SET["low_phase_margin"], "Agent-facing Bode render", save_path=None)
png_bytes = figure_to_png_bytes(figd)
print("PNG payload bytes:", len(png_bytes))

met = extract_gain_phase_margins(
    BODE_SET["low_phase_margin"]["w"],
    BODE_SET["low_phase_margin"]["mag_db"],
    BODE_SET["low_phase_margin"]["phase_deg"],
)
metrics_for_prompt = {**met, "scenario": "low_phase_margin"}
prompt_filled = VISION_ANALYSIS_PROMPT.replace("{{METRICS_JSON}}", json.dumps(metrics_for_prompt, indent=2))
print(prompt_filled.splitlines()[0])

resp = simulate_vision_llm({**metrics_for_prompt, "THD_percent": 1.2, "overshoot": 0.05}, scenario="low_phase_margin")
print(resp)
print(parse_agent_suggestions(resp))

## 10C.6 Full diagnostic loop — simulate → plot → analyze → suggest

We stitch the **tool chain** into a single function `run_diagnostic_once`:

1. **Select** a scenario (Bode or transient).
2. **Compute** numeric metrics (margins / settling / THD).
3. **Render** an annotated figure; optionally **emit PNG** for a vision endpoint.
4. **Fill** the prompt template and call **`simulate_vision_llm`**.
5. **Parse** suggestions into **structured knob deltas**.

This is the **same control flow** you would use with a real **Gemini/Claude** multimodal call—only step 4 is mocked here.

In [ ]:
# --- Single-shot diagnostic loop --------------------------------------------------------

@dataclass
class Knobs:
    I_bias_scale: float = 1.0
    Cc_pf: float = 1.6
    second_pole_factor: float = 1.0  # >1 pushes non-dominant pole outward (toy)


def synthesize_loop_from_knobs(k: Knobs) -> Dict[str, np.ndarray]:
    """Toy mapping from knobs to open-loop Bode (engineering caricature)."""
    w_gc = 2 * np.pi * 12e6 * (k.I_bias_scale**0.35)  # more bias → more GBW (weak)
    A0_db = 80.0
    A0 = 10 ** (A0_db / 20)
    p1 = w_gc / (10 ** (80 / 20))
    p2 = (8 * w_gc) / (k.second_pole_factor * max(0.4, k.Cc_pf / 1.6))
    w, mag, ph = bode_open_loop(A0, [p1, p2], np.logspace(1, 9, 2000))
    ph = add_rhp_zero_effect(ph, w, wz=3 * w_gc / max(0.7, k.I_bias_scale), kappa=0.55)
    return {"w": w, "mag_db": mag, "phase_deg": ph}


def run_diagnostic_once(
    scenario_name: str,
    knobs: Knobs,
    transient_key: Optional[str] = None,
) -> Dict[str, object]:
    pack = synthesize_loop_from_knobs(knobs)
    margins = extract_gain_phase_margins(pack["w"], pack["mag_db"], pack["phase_deg"])
    fig = plot_diagnostic_bode(pack, title=f"Scenario {scenario_name} | knobs={knobs}")
    png = figure_to_png_bytes(fig)

    metrics = {**margins, "I_bias_scale": knobs.I_bias_scale, "Cc_pf": knobs.Cc_pf}
    if transient_key:
        tr = TRANSIENT_SET[transient_key]
        st = settling_time_and_overshoot(tr["t"], tr["y"])
        thd = compute_thd_percent(tr["y"], tr["fs"], f0=50e6)
        metrics.update({**st, **thd})

    prompt = VISION_ANALYSIS_PROMPT.replace("{{METRICS_JSON}}", json.dumps(metrics, indent=2))
    resp = simulate_vision_llm(metrics, scenario=scenario_name)
    parsed = parse_agent_suggestions(resp)
    return {
        "metrics": metrics,
        "png_bytes": png,
        "prompt": prompt,
        "response": resp,
        "parsed": parsed,
    }


demo = run_diagnostic_once("knob_sweep_mid", Knobs(I_bias_scale=1.05, Cc_pf=1.2), transient_key="mild_nonlinearity_thd")
print("Parsed suggestions:", demo["parsed"])
print("PM:", demo["metrics"]["phase_margin_deg"])

## 10C.7 Iterative debug loop — convergence and regression tracking

We now iterate **simulate → analyze → suggest → apply knobs → re-simulate** for $t=1,\dots,T$.

**Knob update (pedagogical, stable):**
$$\theta_{t+1} = \theta_t + \eta \cdot (\theta^\star_{\text{LLM},t} - \theta_t), \quad \eta \in (0,1].$$

We log **PM**, **GM**, **THD**, and **overshoot** (when transients are evaluated) to visualize **progress**. A **plotly** dark chart summarizes **convergence**.

> **Takeaway:** Multimodal debug agents are only as trustworthy as the **numeric validators** they chain with—vision proposes, **metrics dispose**.

In [ ]:
# --- Multi-iteration closed-loop debug --------------------------------------------------


def apply_parsed_to_knobs(k: Knobs, parsed: Dict[str, object], eta: float = 0.55) -> Knobs:
    if not parsed.get("ok"):
        return k
    ib = float(parsed["I_BIAS_SCALE"])
    cc = float(parsed["CC_PF"])
    return Knobs(
        I_bias_scale=(1 - eta) * k.I_bias_scale + eta * ib,
        Cc_pf=(1 - eta) * k.Cc_pf + eta * cc,
        second_pole_factor=min(2.2, k.second_pole_factor * (1 + 0.05 * (cc / k.Cc_pf - 1))),
    )


def iterative_debug(
    init: Knobs,
    steps: int = 6,
    eta: float = 0.55,
    transient_key: str = "excessive_overshoot",
) -> Dict[str, object]:
    k = init
    hist: List[Dict[str, float]] = []
    for it in range(steps):
        pack = synthesize_loop_from_knobs(k)
        m = extract_gain_phase_margins(pack["w"], pack["mag_db"], pack["phase_deg"])
        tr = TRANSIENT_SET[transient_key]
        st = settling_time_and_overshoot(tr["t"], tr["y"])
        thd = compute_thd_percent(tr["y"], tr["fs"], f0=50e6)
        metrics = {
            "iter": float(it),
            "PM": m["phase_margin_deg"],
            "GM": m["gain_margin_dB"],
            "THD": thd["THD_percent"],
            "overshoot": st["overshoot"],
            "I_bias_scale": k.I_bias_scale,
            "Cc_pf": k.Cc_pf,
        }
        hist.append(metrics)
        resp = simulate_vision_llm({**metrics, **m, **st, **thd}, scenario="iterative_tune")
        parsed = parse_agent_suggestions(resp)
        k = apply_parsed_to_knobs(k, parsed, eta=eta)
    return {"knobs_final": k, "history": hist}


trace = iterative_debug(Knobs(I_bias_scale=1.15, Cc_pf=1.0, second_pole_factor=0.85), steps=7, eta=0.5)
hist = trace["history"]
iters = [h["iter"] for h in hist]
fig = go.Figure()
fig.add_trace(go.Scatter(x=iters, y=[h["PM"] for h in hist], mode="lines+markers", name="PM (deg)"))
fig.add_trace(go.Scatter(x=iters, y=[h["GM"] for h in hist], mode="lines+markers", name="GM (dB)", yaxis="y2"))
fig.add_trace(go.Scatter(x=iters, y=[h["THD"] for h in hist], mode="lines+markers", name="THD (%)", yaxis="y3"))
fig.update_layout(
    title="Iterative multimodal debug — metrics vs. iteration (simulated vision)",
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    plot_bgcolor="#11151c",
    xaxis=dict(title="Iteration"),
    yaxis=dict(title="Phase margin (°)", color=GREEN),
    yaxis2=dict(title="Gain margin (dB)", overlaying="y", side="right", color=ACCENT, showgrid=False),
    yaxis3=dict(title="THD (%)", overlaying="y", side="right", anchor="free", position=0.95, color=PURPLE, showgrid=False),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    margin=dict(t=70, r=70),
    height=480,
)
fig.show()

print("Final knobs:", trace["knobs_final"])

## 10C.8 Summary

- **Visual inspection** is not aesthetic—it encodes **high-dimensional** stability and linearity cues that **multimodal LLMs** can operationalize when paired with **rasterized** verification plots.
- A **numeric analysis engine** (margins, settling, THD) provides **ground truth** for **simulated** or **real** vision models.
- **Structured prompts** and **strict trailers** make **suggestions parseable**, enabling **closed-loop** EDA agents.

**Extensions:** hook `simulate_vision_llm` to a provider vision API; replace `synthesize_loop_from_knobs` with **Ngspice/Xyce** subprocess calls; add **corner envelopes** and **Monte Carlo** ribbons on the same axes for **robust** multimodal reasoning.